In [ ]:
!pip install google-genai chromadb sentence-transformers duckduckgo-search pandas requests --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [ ]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
import requests
from datetime import datetime
from google import genai
from google.genai import types
from duckduckgo_search import DDGS

# ==========================================
# 1. CONFIGURATION & API KEYS
# ==========================================
GEMINI_API_KEY = "your_api_key"
WEATHER_API_KEY = "your_api_key" # Using OpenWeather logic from your file

client = genai.Client(api_key=GEMINI_API_KEY)

# ==========================================
# 2. VECTOR DATABASE SETUP (ChromaDB)
# ==========================================
# This replaces standard CSV reading with Vector Search
chroma_client = chromadb.Client()
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Create collections for your datasets, or get them if they already exist
places_collection = chroma_client.get_or_create_collection(name="indian_places", embedding_function=embed_fn)
cost_collection = chroma_client.get_or_create_collection(name="travel_costs", embedding_function=embed_fn)

def ingest_data():
    # Load your CSVs
    p_df = pd.read_csv("Top Indian Places to Visit.csv").fillna("")
    c_df = pd.read_csv("travel cost.csv").fillna("")

    # Debugging: Print column names to check for 'Place'
    print("Columns in Top Indian Places to Visit.csv:", p_df.columns)

    # Ingest Places
    places_collection.add(
        documents=[f"Place: {r['Name']}, City: {r['City']}, Type: {r['Type']}, Description: {r['Significance']}" for _, r in p_df.iterrows()],
        metadatas=[{"city": r['City'], "fee": str(r.get('Entrance Fee in INR', 0))} for _, r in p_df.iterrows()],
        ids=[f"p_{i}" for i in range(len(p_df))]
    )

    # Ingest Costs
    cost_collection.add(
        documents=[f"City: {r['City']}, Typical Stay Cost: {r.iloc[-1]}" for _, r in c_df.iterrows()],
        metadatas=[{"city": r['City']} for _, r in c_df.iterrows()],
        ids=[f"c_{i}" for i in range(len(c_df))]
    )

ingest_data()

# ==========================================
# 3. DEFINING AGENT TOOLS (Functions)
# ==========================================

def get_weather(city: str):
    """Fetches real-time weather for a city."""
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={WEATHER_API_KEY}&units=metric"
    res = requests.get(url).json()
    if "main" not in res: return f"Weather data not found for {city}."
    return f"Weather in {city}: {res['weather'][0]['description']}, Temp: {res['main']['temp']}°C, Humidity: {res['main']['humidity']}%"

def calculate_travel_budget(city: str, days: int, user_budget_per_day: int):
    """Calculates a detailed budget breakdown based on local data."""
    # Query Vector DB for city-specific base costs
    cost_result = cost_collection.query(query_texts=[city], n_results=1)

    # Logic extracted from your Budget_Logic.ipynb
    base_hotel = 1500 # Default
    if cost_result['documents']:
        try: base_hotel = int(cost_result['documents'][0][0].split(':')[-1].split('-')[0].strip())
        except: pass

    food = 500
    transport = 300
    sightseeing = 400

    base_total = base_hotel + food + transport + sightseeing
    scale = user_budget_per_day / base_total

    return {
        "daily_breakdown": {
            "Hotel": int(base_hotel * scale),
            "Food": int(food * scale),
            "Transport": int(transport * scale),
            "Sightseeing": int(sightseeing * scale)
        },
        "total_estimate": int(user_budget_per_day * days)
    }

def get_local_recommendations(city: str):
    """Retrieves top places to visit from the vector database."""
    results = places_collection.query(query_texts=[f"best places in {city}"], n_results=5)
    return results['documents'][0]

def web_search_discovery(query: str):
    """Uses DuckDuckGo to find hidden gems or recent travel news."""
    with DDGS() as ddgs:
        results = [r['body'] for r in ddgs.text(query, max_results=3)]
    return "\n".join(results)

# ==========================================
# 4. THE GEMINI "BRAIN" INTEGRATION
# ==========================================

tools = [get_weather, calculate_travel_budget, get_local_recommendations, web_search_discovery]

def travel_concierge():
    print("🤖 AI Travel Concierge Active")
    place = input("🌍 Where to? ")
    days = int(input("📅 How many days? "))
    budget = int(input("💰 Daily budget (₹)? "))

    prompt = f"""
    You are a professional Travel Concierge. A user wants to visit {place} for {days} days
    with a daily budget of ₹{budget}.

    Follow these steps:
    1. Check the weather for {place}.
    2. Get local recommendations from our database.
    3. Calculate the budget breakdown.
    4. Use web search to find one 'hidden gem' or recent tip for {place}.
    5. Create a structured itinerary.

    Answer ONLY using the provided tool outputs.
    """

    # Chat session with automatic function calling
    chat = client.chats.create(
        model="gemini-2.0-flash", # Use the latest flash model for speed and tool handling
        config=types.GenerateContentConfig(tools=tools)
    )

    response = chat.send_message(prompt)

    print("\n" + "="*50)
    print(response.text)
    print("="*50)

if __name__ == "__main__":
    travel_concierge()

Columns in Top Indian Places to Visit.csv: Index(['Unnamed: 0', 'Zone', 'State', 'City', 'Name', 'Type',
       'Establishment Year', 'time needed to visit in hrs',
       'Google review rating', 'Entrance Fee in INR',
       'Airport with 50km Radius', 'Weekly Off', 'Significance',
       'DSLR Allowed', 'Number of google review in lakhs',
       'Best Time to visit'],
      dtype='object')
🤖 AI Travel Concierge Active
🌍 Where to? Jaipur
📅 How many days? 6
💰 Daily budget (₹)? 10000


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 31.5098942s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goog